# Notebook 08 — Phase 5: Stratified Residual Analysis

**Purpose.** Phase 5 (ChemProp D-MPNN) finished with 5-seed scaffold test RMSE = 1.148 ± 0.030. This notebook checks whether the *shape* of the error has changed relative to Phase 4 (MLP Morgan), by stratifying residuals on:

1. **logS regime** — hydrophilic (logS ≥ −1), moderate (−4 < logS < −1), hydrophobic (logS ≤ −4)
2. **Polyol substructure** — SMARTS pattern with ≥3 hydroxyl groups on sp3 C
3. **Top-10 worst residuals** — same dedup-by-canonical-SMILES protocol as `07_phase4_worst10_analysis.ipynb`

**Pre-registered targets** (from the project context, set before running ChemProp):

| Pattern             | Phase 4 baseline | Phase 5 target  |
|---------------------|-----------------:|----------------:|
| hydrophilic mean_resid | −0.674          | toward \|0.1\|   |
| hydrophobic mean_resid | +1.81           | toward \|0.5\|   |
| polyol-aromatic hybrids | ≈ +1.5          | toward \|0.5\|   |

All predictions are read from `reports/phase5_summary.json` — no model re-training needed.

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/qsar-esol-solubility')
else:
    PROJECT_ROOT = Path.cwd().parent

REPORTS_DIR = PROJECT_ROOT / 'reports'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw, inchi
RDLogger.DisableLog('rdApp.*')

print(f'Project root: {PROJECT_ROOT}')

## 2. Load Phase 5 predictions + reconstruct per-molecule DataFrame

The summary JSON has per-seed `test_smiles`, `test_y_true`, `test_y_pred`. Pool them, dedup by canonical SMILES, average predictions across appearances.

In [ ]:
with open(REPORTS_DIR / 'phase5_summary.json') as fh:
    p5 = json.load(fh)

print(f"Phase: {p5['phase']}")
print(f"Aggregate: RMSE={p5['aggregate']['RMSE_mean']:.3f} ± {p5['aggregate']['RMSE_std']:.3f}")
print(f"Number of seeds: {len(p5['per_seed'])}")

# Pool all (seed, smiles, y_true, y_pred) tuples
rows = []
for entry in p5['per_seed']:
    seed = entry['seed']
    for smi, yt, yp in zip(entry['test_smiles'], entry['test_y_true'], entry['test_y_pred']):
        rows.append({
            'seed': seed,
            'smiles': smi,
            'y_true': float(yt),
            'y_pred': float(yp),
            'residual': float(yp - yt),
            'abs_residual': float(abs(yp - yt)),
        })

preds_df = pd.DataFrame(rows)
print(f'\nPool: {len(preds_df)} predictions over {preds_df["smiles"].nunique()} distinct molecules')
print(f'Pooled RMSE: {np.sqrt((preds_df["residual"]**2).mean()):.4f}')

## 3. Stratification by logS regime

Same cutoffs used in Phase 4 stratified analysis:
- **hydrophilic**: y_true ≥ −1
- **moderate**: −4 < y_true < −1
- **hydrophobic**: y_true ≤ −4

In [ ]:
def assign_regime(logS):
    if logS >= -1: return 'hydrophilic'
    if logS <= -4: return 'hydrophobic'
    return 'moderate'

preds_df['regime'] = preds_df['y_true'].apply(assign_regime)

regime_stats = (
    preds_df.groupby('regime')
    .agg(n=('residual', 'size'),
         mean_resid=('residual', 'mean'),
         std_resid=('residual', 'std'),
         rmse=('residual', lambda r: float(np.sqrt(np.mean(r**2)))),
         mae=('abs_residual', 'mean'))
    .reindex(['hydrophilic', 'moderate', 'hydrophobic'])
)
print('Phase 5 — residuals by logS regime:')
print(regime_stats.round(3).to_string())

In [ ]:
# Compare to Phase 4 baseline (from project context)
p4_baseline = {
    'hydrophilic': -0.674,
    'moderate':    +0.55,
    'hydrophobic': +1.81,
}
p5_targets = {
    'hydrophilic': 0.10,   # |target| = 0.10
    'moderate':    0.30,   # implicit, not pre-registered tightly
    'hydrophobic': 0.50,
}

print(f"\n{'Regime':<12} {'P4 (MLP)':>10} {'P5 (ChemP)':>11} {'Target':>9} {'Verdict':>15}")
print('-' * 65)
for regime in ['hydrophilic', 'moderate', 'hydrophobic']:
    p4 = p4_baseline[regime]
    p5 = float(regime_stats.loc[regime, 'mean_resid'])
    target = p5_targets[regime]
    if abs(p5) <= target:
        verdict = '✓ ON target'
    elif abs(p5) < abs(p4):
        verdict = '~ improved'
    else:
        verdict = '✗ no improvement'
    print(f"{regime:<12} {p4:>+10.3f} {p5:>+11.3f} {target:>9.2f} {verdict:>15}")

## 4. Polyol diagnostic (≥3 OH on sp3 C)

Same SMARTS as Phase 4 diagnostic: `[CX4]([OX2H])[CX4]([OX2H])[CX4]([OX2H])` matches any three contiguous sp3 carbons each bearing a hydroxyl. This catches sugars (glucose, sucrose), polyols (mannitol, sorbitol), and polyol-aromatic hybrids (riboflavin, glycosylated flavones).

In [ ]:
POLYOL_SMARTS = '[CX4]([OX2H])[CX4]([OX2H])[CX4]([OX2H])'
patt = Chem.MolFromSmarts(POLYOL_SMARTS)

def is_polyol(smi):
    m = Chem.MolFromSmiles(smi)
    return m is not None and m.HasSubstructMatch(patt)

preds_df['is_polyol'] = preds_df['smiles'].apply(is_polyol)

n_polyol = preds_df['is_polyol'].sum()
polyol_stats = (
    preds_df[preds_df['is_polyol']]
    .agg({'residual': ['count', 'mean', 'std'],
          'abs_residual': 'mean'})
)
print(f'Polyol diagnostic ({POLYOL_SMARTS}):')
print(f'  Matches in pool : {n_polyol} predictions over {preds_df[preds_df["is_polyol"]]["smiles"].nunique()} distinct molecules')
if n_polyol > 0:
    print(f'  mean residual   : {preds_df[preds_df["is_polyol"]]["residual"].mean():+.3f}')
    print(f'  std residual    : {preds_df[preds_df["is_polyol"]]["residual"].std():.3f}')
    print(f'  mean |residual| : {preds_df[preds_df["is_polyol"]]["abs_residual"].mean():.3f}')
    print()
    print('Compare to Phase 4 polyol baseline:')
    print('  P4 mean residual : +0.488 (over-prediction of solubility)')
    print(f'  P5 mean residual : {preds_df[preds_df["is_polyol"]]["residual"].mean():+.3f}')
    p5_polyol = preds_df[preds_df['is_polyol']]['residual'].mean()
    if abs(p5_polyol) <= 0.5:
        print(f'  → ON target (|residual| ≤ 0.5)')
    elif abs(p5_polyol) < abs(0.488):
        print(f'  → ~ improved over P4 but not on target')
    else:
        print(f'  → not improved vs P4')

## 5. Top-10 worst residuals — distinct molecules (same protocol as notebook 07)

In [ ]:
# Dedup by canonical SMILES, aggregate residual stats
preds_df['canonical'] = preds_df['smiles'].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s))
)

mol_summary = (
    preds_df
    .groupby('canonical', as_index=False)
    .agg(
        smiles=('smiles', 'first'),
        y_true=('y_true', 'first'),
        y_pred_mean=('y_pred', 'mean'),
        y_pred_std=('y_pred', 'std'),
        abs_residual_mean=('abs_residual', 'mean'),
        n_appearances=('seed', 'count'),
        seeds_appeared=('seed', lambda s: sorted(s.tolist())),
    )
)

top10 = (
    mol_summary
    .sort_values('abs_residual_mean', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10['residual'] = top10['y_pred_mean'] - top10['y_true']

print('Phase 5 — Top-10 worst DISTINCT molecules:')
print('-' * 100)
for i, r in enumerate(top10.itertuples(index=False), start=1):
    print(f'#{i:>2}  y_true={r.y_true:+.2f}  y_pred={r.y_pred_mean:+.2f}  '
          f'res={r.residual:+.2f}  n={r.n_appearances}/5  seeds={r.seeds_appeared}')
    print(f'      {r.smiles}')

## 6. Cross-phase comparison of worst-10 molecules

Do Phase 4 and Phase 5 fail on the *same* molecules? Or has the architecture shift changed which molecules are problematic?

Loads the Phase 4 worst-10 list from `reports/phase4_worst10.json` (produced by notebook 07) and compares by canonical SMILES.

In [ ]:
p4_worst10_path = REPORTS_DIR / 'phase4_worst10.json'
if p4_worst10_path.exists():
    with open(p4_worst10_path) as fh:
        p4_worst10 = json.load(fh)
    p4_canonical = {
        Chem.MolToSmiles(Chem.MolFromSmiles(entry['smiles'])): entry
        for entry in p4_worst10['top10']
    }
    p5_canonical = set(top10['canonical'])

    common = set(p4_canonical) & p5_canonical
    p4_only = set(p4_canonical) - p5_canonical
    p5_only = p5_canonical - set(p4_canonical)

    print(f'Worst-10 overlap between Phase 4 and Phase 5:')
    print(f'  Common to both : {len(common)} / 10')
    print(f'  In P4 only     : {len(p4_only)}')
    print(f'  In P5 only     : {len(p5_only)}')
    print()
    print('Common molecules (failure mode persists across architectures):')
    for smi in common:
        p4_resid = p4_canonical[smi]['residual']
        p5_row = top10[top10['canonical'] == smi].iloc[0]
        print(f'  {smi[:60]:<60}  P4_res={p4_resid:+.2f}  P5_res={p5_row["residual"]:+.2f}')
else:
    print(f'phase4_worst10.json not found at {p4_worst10_path}')
    print('  Skipping cross-phase comparison. Run notebook 07 first.')

## 7. Persist results

In [ ]:
out = {
    'phase': 'phase5_chemprop_dmpnn',
    'analysis': 'stratified_residuals',
    'pool_size': int(len(preds_df)),
    'n_distinct_molecules': int(preds_df['smiles'].nunique()),
    'regime_stats': regime_stats.round(4).to_dict('index'),
    'polyol': {
        'n_predictions':       int(n_polyol),
        'n_distinct_molecules': int(preds_df[preds_df['is_polyol']]['smiles'].nunique()),
        'mean_residual':       float(preds_df[preds_df['is_polyol']]['residual'].mean()) if n_polyol > 0 else None,
        'std_residual':        float(preds_df[preds_df['is_polyol']]['residual'].std()) if n_polyol > 0 else None,
    },
    'top10': [
        {
            'rank': i + 1,
            'smiles': r.smiles,
            'canonical_smiles': r.canonical,
            'y_true': r.y_true,
            'y_pred_mean': r.y_pred_mean,
            'y_pred_std': r.y_pred_std if pd.notna(r.y_pred_std) else None,
            'residual': r.residual,
            'n_appearances': int(r.n_appearances),
            'seeds_appeared': list(r.seeds_appeared),
        }
        for i, r in enumerate(top10.itertuples(index=False))
    ],
}

out_path = REPORTS_DIR / 'phase5_stratified.json'
out_path.write_text(json.dumps(out, indent=2, default=str))
print(f'Saved: {out_path}')

## 8. Verdict summary

Decision rule pre-registered in the project context: if ChemProp hits all three targets (hydrophilic, hydrophobic, polyol), the graph representation is empirically validated. If it hits the first two but not the third, the polyol-aromatic case is a 3D/solvation problem out of reach of 2D graphs. Both outcomes are informative.

In [ ]:
# Auto-emit verdict
h_p5 = float(regime_stats.loc['hydrophilic', 'mean_resid'])
o_p5 = float(regime_stats.loc['hydrophobic', 'mean_resid'])
pol_p5 = float(preds_df[preds_df['is_polyol']]['residual'].mean()) if n_polyol > 0 else None

h_ok = abs(h_p5) <= 0.10
o_ok = abs(o_p5) <= 0.50
pol_ok = pol_p5 is not None and abs(pol_p5) <= 0.50

print('=== Pre-registered verdict ===')
print()
print(f"  Target 1 — hydrophilic |resid| ≤ 0.10:  {'✓' if h_ok else '✗'}  ({h_p5:+.3f})")
print(f"  Target 2 — hydrophobic |resid| ≤ 0.50:  {'✓' if o_ok else '✗'}  ({o_p5:+.3f})")
if pol_p5 is not None:
    print(f"  Target 3 — polyol     |resid| ≤ 0.50:  {'✓' if pol_ok else '✗'}  ({pol_p5:+.3f})")
else:
    print(f'  Target 3 — polyol: no matches in pool, cannot evaluate.')

print()
if h_ok and o_ok and pol_ok:
    print('→ STRONG: graph representation empirically validated. All three targets hit.')
elif h_ok and o_ok and not pol_ok:
    print('→ PARTIAL: hydrophilic + hydrophobic resolved, polyol-aromatic case persists.')
    print('  Interpretation: 2D graph alone insufficient for polyol-aromatic hybrids;')
    print('  likely a 3D/solvation problem out of reach of 2D representations.')
else:
    print('→ WEAK: graph representation does not resolve the compression bias.')
    print('  Confirms that compression bias at the extremes is feature-class,')
    print('  not curable by simply switching from Morgan FP to learned graph features.')